In [13]:
import json
import os
import pandas as pd
import re
import torch

from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
from transformers import pipeline, BitsAndBytesConfig

In [2]:
BATCH_SIZE = 8
MAX_NEW_TOKENS = 20
DO_SAMPLE = False
NUM_BEAMS = 1

PROJECT_PATH = Path(os.getcwd()).parent.parent
DS_PATH = PROJECT_PATH / "datasets" / "fine-tuning" / "conversational"
TESTING_DS_PATH = DS_PATH / "ordered_testing_ds_juliet.jsonl"

OUTPUT_METRICS_PATH = Path(os.getcwd()).parent.parent / "output_metrics"
OUTPUT_METRICS_FILE = OUTPUT_METRICS_PATH / "falcon3-1b-instruct.txt"
MODEL_ID = str(Path("~/ft/falcon3-1b-instruct/fine-tuned").expanduser())

#MODEL_ID = "tiiuae/Falcon3-1B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [4]:
pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    device=DEVICE,
    model_kwargs={
        "quantization_config": quantization_config,
        "dtype": torch.bfloat16,
    }
)

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading weights:   0%|          | 0/72 [00:00<?, ?it/s]

In [5]:
if pipe.tokenizer.pad_token is None:
    print("WARNING: No pad token defined. Using end of sentence token.")
    pipe.tokenizer.pad_token = pipe.tokenizer.eos_token

In [6]:
def extract_prompt(example):
    return example["messages"]

In [7]:
examples = []
with open(TESTING_DS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        examples.append(json.loads(line.strip()))

first_5K_cases = examples[:5000]

In [8]:
len(first_5K_cases)

5000

In [9]:
generated_texts = []
ground_truths = []

In [10]:
print(f"Processing {len(first_5K_cases)} examples in batches of size {BATCH_SIZE}...")
## Load in tqdm to see progress bar
for i in tqdm(range(0, len(first_5K_cases), BATCH_SIZE), desc="Procesando"):
    batch_examples = first_5K_cases[i:i + BATCH_SIZE]
    prompts = [extract_prompt(example) for example in batch_examples]
    true_labels = [example.get("cwe", []) for example in batch_examples]

    with torch.no_grad():
        outputs = pipe(
            prompts,
            max_new_tokens=MAX_NEW_TOKENS,
            batch_size=BATCH_SIZE,
            padding=True,
            truncation = True,
            return_full_text=False,
            do_sample=DO_SAMPLE,
            num_beams=NUM_BEAMS,
        )

    for k, (output, true_label) in enumerate(zip(outputs, true_labels)):
        generated = output[0]["generated_text"] if isinstance(output, list) else output["generated_text"]

        generated_clean = generated.strip()
        generated_texts.append(generated_clean)
        ground_truths.append(true_label)

Processing 5000 examples in batches of size 8...


Procesando:   0%|          | 0/625 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'num_beams', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Procesando:   2%|▏         | 10/625 [00:13<13:13,  1.29s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refe

In [14]:
regex_cwe = re.compile(r"CWE-\d{1,3}", re.IGNORECASE)
regex_safe = re.compile(r"SAFE", re.IGNORECASE)

predictions = []
for text in generated_texts:
    match_cwe = regex_cwe.search(text)
    if match_cwe:
        predictions.append(match_cwe.group(0))
        continue

    match_safe = regex_safe.search(text)
    if match_safe:
        predictions.append(match_safe.group(0))
        continue

    predictions.append("UNKNOWN")

In [16]:
valid_mask = [p != "ERROR" and t != "ERROR" for p, t in zip(predictions, ground_truths)]
valid_pred = [p for p, v in zip(predictions, valid_mask) if v]
valid_true = [t for t, v in zip(ground_truths, valid_mask) if v]

labels = sorted(set(valid_true + valid_pred))
print("Etiquetas encontradas:", labels)

cm = confusion_matrix(valid_true, valid_pred)

# CREATE OUTPUT METRICS FILE DIRECTORY
OUTPUT_METRICS_PATH.mkdir(exist_ok=True)
with open(OUTPUT_METRICS_FILE, 'w') as f:
    print("\nConfusion Matrix:", file=f)
    print(pd.DataFrame(cm, index=labels, columns=labels), file=f)

    print("\nClassification Report:", file=f)
    print(classification_report(valid_true, valid_pred), file=f)


Etiquetas encontradas: ['CWE-120', 'CWE-121', 'CWE-122', 'CWE-123', 'CWE-124', 'CWE-126', 'CWE-127', 'CWE-15', 'CWE-401', 'CWE-415', 'CWE-416', 'CWE-476', 'CWE-562', 'CWE-590', 'CWE-68', 'CWE-680', 'CWE-685', 'CWE-688', 'CWE-761', 'CWE-78', 'CWE-789', 'CWE-835', 'CWE-89', 'SAFE', 'UNKNOWN']


C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf